## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI:

```bash
az login
```

# 💬 Chat Middleware

## Industry Use Case: Customer Service Message Filtering

This notebook demonstrates **chat middleware** for intercepting and modifying messages.

| Feature | FSI Application |
|---------|-----------------|
| **Message Observation** | Audit logging for compliance |
| **Message Modification** | PII redaction |
| **Response Override** | Block sensitive queries |

In [7]:
import os
from dotenv import load_dotenv

load_dotenv('../../.env', override=True)
load_dotenv("../../.env", override=True)

PROJECT_ENDPOINT = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")


print(f"✅ Environment loaded")

✅ Environment loaded


In [8]:
from collections.abc import Awaitable, Callable

from agent_framework import (
    Agent,
    ChatContext,
    ChatMiddleware,
    ChatResponse,
    Message,
    MiddlewareTermination,
    chat_middleware,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print("✅ All imports loaded")

✅ All imports loaded


## Define FSI Tool Function

In [9]:
def get_account_balance(account_id: str) -> str:
    """Get account balance for a customer."""
    return f"Account {account_id} has a balance of $15,432.50"

print("✅ Tool defined: get_account_balance")

✅ Tool defined: get_account_balance


## Define Chat Middleware

Chat middleware intercepts requests before they reach the AI service.

In [10]:
# Class-based middleware for message observation
class InputObserverMiddleware(ChatMiddleware):
    """Observes and logs input messages."""

    async def process(
        self,
        context: ChatContext,
        next: Callable[[], Awaitable[None]],
    ) -> None:
        print("[Observer] Observing input messages:")
        for i, message in enumerate(context.messages):
            content = message.text if message.text else str(message.contents)
            print(f"  Message {i + 1} ({message.role}): {content[:50]}...")
        
        await next()
        print("[Observer] Processing completed")


# Function-based middleware for security filtering
@chat_middleware
async def security_middleware(
    context: ChatContext,
    next: Callable[[], Awaitable[None]],
) -> None:
    """Blocks sensitive information requests."""
    blocked_terms = ["password", "secret", "ssn", "social security"]

    for message in context.messages:
        if message.text:
            for term in blocked_terms:
                if term in message.text.lower():
                    print(f"[Security] BLOCKED: Found '{term}'")
                    context.result = ChatResponse(
                        messages=[Message(role="assistant", contents=["I cannot process requests for sensitive information."])]
                    )
                    raise MiddlewareTermination()

    await next()

print("✅ Middleware defined: InputObserverMiddleware, security_middleware")

✅ Middleware defined: InputObserverMiddleware, security_middleware


## Example 1: Class-based Chat Middleware

In [11]:
async def class_based_example():
    print("=== Class-based Chat Middleware ===")
    
    credential = AzureCliCredential()
    client = FoundryChatClient(
        project_endpoint=PROJECT_ENDPOINT,
        model=MODEL_DEPLOYMENT,
        credential=credential,
    )
    agent = Agent(
        client=client,
        name="ObserverAgent",
        instructions="You are a helpful banking assistant.",
        middleware=[InputObserverMiddleware()],
        tools=get_account_balance,
    )

    query = "What is the balance for account ACC-123?"
    print(f"User: {query}")
    result = await agent.run(query)
    print(f"Agent: {result.text if result.text else 'No response'}")

await class_based_example()

=== Class-based Chat Middleware ===


User: What is the balance for account ACC-123?
[Observer] Observing input messages:
  Message 1 (user): What is the balance for account ACC-123?...
[Observer] Processing completed
[Observer] Observing input messages:
  Message 1 (tool): [<agent_framework._types.Content object at 0x00000...
[Observer] Processing completed
Agent: The balance for account ACC-123 is $15,432.50. If you need more details or transactions, please let me know!


## Example 2: Function-based Security Middleware

In [12]:
async def security_example():
    print("=== Security Middleware ===")
    
    credential = AzureCliCredential()
    client = FoundryChatClient(
        project_endpoint=PROJECT_ENDPOINT,
        model=MODEL_DEPLOYMENT,
        credential=credential,
    )
    agent = Agent(
        client=client,
        name="SecureAgent",
        instructions="You are a helpful assistant.",
        middleware=[security_middleware],
    )

    # Normal query
    print("\n--- Normal Query ---")
    query = "Hello, how are you?"
    print(f"User: {query}")
    result = await agent.run(query)
    print(f"Agent: {result.text if result.text else 'No response'}")

    # Blocked query
    print("\n--- Blocked Query ---")
    query = "What is my password?"
    print(f"User: {query}")
    result = await agent.run(query)
    print(f"Agent: {result.text if result.text else 'No response'}")

await security_example()

=== Security Middleware ===

--- Normal Query ---
User: Hello, how are you?
Agent: Hello! I'm doing well, thank you for asking. How can I help you today?

--- Blocked Query ---
User: What is my password?
[Security] BLOCKED: Found 'password'
Agent: I cannot process requests for sensitive information.


## Example 3: Run-level Middleware

In [13]:
async def run_level_example():
    print("=== Run-level Middleware ===")
    
    credential = AzureCliCredential()
    client = FoundryChatClient(
        project_endpoint=PROJECT_ENDPOINT,
        model=MODEL_DEPLOYMENT,
        credential=credential,
    )
    agent = Agent(
        client=client,
        name="FlexibleAgent",
        instructions="You are a helpful assistant.",
        # No middleware at agent level
    )

    # Run without middleware
    print("\n--- Without Middleware ---")
    query = "Hello"
    print(f"User: {query}")
    result = await agent.run(query)
    print(f"Agent: {result.text if result.text else 'No response'}")

    # Run with middleware for this call only
    print("\n--- With Run-level Middleware ---")
    query = "What is my SSN?"
    print(f"User: {query}")
    result = await agent.run(query, middleware=[security_middleware])
    print(f"Agent: {result.text if result.text else 'No response'}")

await run_level_example()

=== Run-level Middleware ===

--- Without Middleware ---
User: Hello
Agent: Hello! How can I help you today?

--- With Run-level Middleware ---
User: What is my SSN?
[Security] BLOCKED: Found 'ssn'
Agent: I cannot process requests for sensitive information.


## 📋 Key Takeaways

| Feature | Description |
|---------|-------------|
| **ChatMiddleware class** | Class-based middleware with `process()` method |
| **@chat_middleware** | Function decorator for chat middleware |
| **Agent-level** | Middleware applies to all runs |
| **Run-level** | Middleware applies to specific run only |
| **context.terminate** | Stop execution and return override response |